# ASR RTF Benchmark — ChunkFormer-CTC-Large-Vie

- **Model**: `khanhld/chunkformer-ctc-large-vie` (110M params, trained on 3000h)  
- **Dataset**: 6000 WAV từ `doof-ferb/vlsp2020_vinai_100h`
- **Metrics**: RTF P50, RTF P95
- **Output**: `Drive/asr_benchmark/asr_rtf_results/chunkformer_ctc_large.jsonl` + `_summary.json`

Output format tương thích với `eval_asr_rtf.ipynb` để gộp so sánh sau.

## Trước khi chạy

Upload lên Google Drive:
```
Drive/asr_benchmark/samples/                    ← 6000 WAV
Drive/asr_benchmark/asr_benchmark_samples.jsonl
```
Model tự download từ HuggingFace — không cần upload weights.  
Chọn Runtime: **A100 GPU**

## RTF
```
RTF = processing_time / audio_duration
```

In [ ]:
!pip install -q chunkformer soundfile numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, time
from pathlib import Path
import numpy as np

DRIVE_DIR   = '/content/drive/MyDrive/asr_benchmark'
SAMPLES_DIR = Path(f'{DRIVE_DIR}/samples')
JSONL_PATH  = f'{DRIVE_DIR}/asr_benchmark_samples.jsonl'
OUT_DIR     = Path(f'{DRIVE_DIR}/asr_rtf_results')
OUT_DIR.mkdir(exist_ok=True)

samples = []
with open(JSONL_PATH, encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))

print(f'Loaded {len(samples)} samples')
durations = [s['duration'] for s in samples]
print(f'Duration — min={min(durations):.1f}s  mean={np.mean(durations):.1f}s  max={max(durations):.1f}s')
print(f'Total   : {sum(durations)/60:.1f} min audio')

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# Load ChunkFormer model
from chunkformer import ChunkFormerModel

MODEL_NAME = 'chunkformer_ctc_large'
model = ChunkFormerModel.from_pretrained('khanhld/chunkformer-ctc-large-vie')
print('Model loaded.')

In [ ]:
# Warmup + giới hạn 1000 samples
N_WARMUP = 10
N_BENCH  = 1000

print(f'Warming up ({N_WARMUP} samples)...')
for s in samples[:N_WARMUP]:
    model.endless_decode(
        audio_path=str(SAMPLES_DIR / s['wav_file']),
        chunk_size=64, left_context_size=128, right_context_size=128,
        total_batch_duration=1800,
    )
print('Done.')

bench_samples = samples[:N_BENCH]
print(f'Benchmark samples: {len(bench_samples)}')

In [ ]:
# Benchmark
results = []
for i, s in enumerate(bench_samples):
    dur = s['duration']
    wav_path = str(SAMPLES_DIR / s['wav_file'])

    t0 = time.perf_counter()
    out = model.endless_decode(
        audio_path=wav_path,
        chunk_size=64, left_context_size=128, right_context_size=128,
        total_batch_duration=1800,
    )
    elapsed = time.perf_counter() - t0

    text = out if isinstance(out, str) else str(out)
    rtf  = elapsed / dur if dur > 0 else 0.0

    results.append({
        'idx':        s['idx'],
        'wav_file':   s['wav_file'],
        'duration':   dur,
        'latency_ms': round(elapsed * 1000, 2),
        'rtf':        round(rtf, 4),
        'transcript': text,
        'ref':        s['sentence'],
    })

    if (i + 1) % 100 == 0:
        rtfs_so_far = [r['rtf'] for r in results]
        print(f'[{i+1}/{len(bench_samples)}]  RTF P50={np.median(rtfs_so_far):.4f}  P95={np.percentile(rtfs_so_far,95):.4f}')

print('\nBenchmark complete.')

In [ ]:
# Save per-sample JSONL
out_jsonl = OUT_DIR / f'{MODEL_NAME}.jsonl'
with open(out_jsonl, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Saved: {out_jsonl}')

In [ ]:
# Summary
rtfs = [r['rtf'] for r in results]
lats = [r['latency_ms'] for r in results]
durs = [r['duration'] for r in results]

summary = {
    'model':     MODEL_NAME,
    'n_samples': len(results),
    'rtf': {
        'mean':   round(float(np.mean(rtfs)),           4),
        'median': round(float(np.median(rtfs)),         4),
        'p95':    round(float(np.percentile(rtfs, 95)), 4),
        'p99':    round(float(np.percentile(rtfs, 99)), 4),
        'max':    round(float(np.max(rtfs)),            4),
    },
    'latency_ms': {
        'mean':   round(float(np.mean(lats)),           1),
        'median': round(float(np.median(lats)),         1),
        'p95':    round(float(np.percentile(lats, 95)), 1),
    },
    'audio_duration_s': {
        'mean':  round(float(np.mean(durs)),  2),
        'total': round(float(np.sum(durs)),   1),
    },
}

out_json = OUT_DIR / f'{MODEL_NAME}_summary.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('=' * 50)
print(f"Model  : {summary['model']}")
print(f"Samples: {summary['n_samples']}")
print(f"RTF    — mean={summary['rtf']['mean']}  median={summary['rtf']['median']}  P95={summary['rtf']['p95']}  P99={summary['rtf']['p99']}")
print(f"Latency— mean={summary['latency_ms']['mean']}ms  P95={summary['latency_ms']['p95']}ms")
print('=' * 50)
print(f'Saved: {out_json}')

In [ ]:
# Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rtfs, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(np.median(rtfs), color='red',    linestyle='--', label=f'P50={np.median(rtfs):.3f}')
axes[0].axvline(np.percentile(rtfs, 95), color='orange', linestyle='--', label=f'P95={np.percentile(rtfs,95):.3f}')
axes[0].set_xlabel('RTF')
axes[0].set_ylabel('Count')
axes[0].set_title(f'RTF Distribution — {MODEL_NAME}')
axes[0].legend()

axes[1].scatter(durs, rtfs, alpha=0.3, s=5)
axes[1].set_xlabel('Audio duration (s)')
axes[1].set_ylabel('RTF')
axes[1].set_title('RTF vs Audio Duration')

plt.tight_layout()
out_png = OUT_DIR / f'{MODEL_NAME}_rtf_dist.png'
plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')